# Practice 5 — Intrinsic dimension

## 0. Setup

In [ ]:
%pip install -q "scikit-dimension==0.3.7" "numpy>=2.1" "scikit-learn>=1.6" matplotlib pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skdim
from importlib.metadata import version
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

SEED = 5

def neighbors(X, k):
    """Distances and indices of k OTHER observations, sorted by distance."""
    if not 2 <= k < len(X):
        raise ValueError("Require 2 <= k < number of observations")
    # X=None excludes the query observation itself, including with tied distances.
    return NearestNeighbors(n_neighbors=k).fit(X).kneighbors()

def check_cloud(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or not np.isfinite(X).all():
        raise ValueError("Expected a finite (samples, features) array")
    if len(np.unique(X, axis=0)) != len(X):
        raise ValueError("Duplicate rows: inspect the sampling process before estimating ID")
    return X

## 1. The manifold hypothesis: $D$ versus $d$
To describe points over the circle in $\mathbb{R}^2$, we need one local coordinate; whereas to describe points over the sphere or over a 2D plane in $\mathbb{R}^3$, we need two parameters. 
We see this in the following examples, where we generate a finite number of points from each manifold and visualize them.
Notice the number of parameters needed to describe the points in each manifold, independently of the ambient space dimension.

In [ ]:
rng = np.random.default_rng(SEED)
n = 1200

# Circle: one parameter, theta
theta = rng.uniform(0, 2*np.pi, n)
circle = np.column_stack([
    np.cos(theta),
    np.sin(theta),
])

# Plane patch: two parameters, u and v
u = rng.uniform(-1, 1, n)
v = rng.uniform(-1, 1, n)
plane = np.column_stack([
    u,
    v,
    np.zeros(n),
])

# Sphere surface: two parameters, theta and phi
# theta is longitude; phi is the angle measured from the north pole.
theta = rng.uniform(0, 2*np.pi, n)
phi = np.arccos(rng.uniform(-1, 1, n))

sphere = np.column_stack([
    np.sin(phi) * np.cos(theta),
    np.sin(phi) * np.sin(theta),
    np.cos(phi),
])

clouds = {
    "Circle": (circle, 1),
    "Plane patch": (plane, 2),
    "Sphere surface": (sphere, 2),
}

In [ ]:
fig = plt.figure(figsize=(12, 3.5))
ax = fig.add_subplot(131)
ax.scatter(*circle.T, s=4) 
ax.set_aspect("equal")
ax.set(title="Circle: D=2, d=1", xlabel=r"$x_1$", ylabel=r"$x_2$")
ax.set_xticks([])
ax.set_yticks([])
for j, (name, X) in enumerate([("Plane patch", plane), ("Sphere surface", sphere)], start=2):
    ax = fig.add_subplot(1, 3, j, projection="3d")
    ax.scatter(*X.T, s=3, alpha=.5)
    ax.set(title=f"{name}: D=3, d=2", xlabel=r"$x_1$", ylabel=r"$x_2$", zlabel=r"$x_3$")
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
plt.tight_layout(); plt.show()

### Optional extension: local volume growth

**Skip to section 2 during the session.** The next two code cells and Try it 1 are optional.
We check experimentally that for locally approximately constant sampling density, the number of points within radius $r$ scales as $N_x(r)\approx C(x)r^d$.
Thus $N_x(2r)/N_x(r)\approx 2^d$. We compare a circumference with a filled disk, which have dimension 1 and 2 respectively.

In [ ]:
rng_growth = np.random.default_rng(51)
theta = rng_growth.uniform(0, 2*np.pi, 15000)
ring = np.c_[np.cos(theta), np.sin(theta)] # uniform distribution over the circumference
radius = np.sqrt(rng_growth.uniform(size=len(theta)))  # sample radius to get uniform density in the disk
disk = radius[:, None] * ring

In [ ]:
radii = np.geomspace(.04, .4, 20) # set limits to exclude the center and the full cloud; use a geometric sequence for log-log plots
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, X, center, d in [("Circle", ring, [1, 0], 1), ("Disk", disk, [0, 0], 2)]: 
    # we count points aroun (1,0) for the circle and around (0,0) for the disk
    distances = np.linalg.norm(X - center, axis=1) # compute distances to the center
    counts = np.array([(distances <= r).sum() for r in radii]) # count points within radius r
    doubled = np.array([(distances <= 2*r).sum() for r in radii]) # count points within radius 2r
    axes[0].loglog(radii, counts, "o-", label=f"{name} (d={d})") # both axes in log scale to visualize the power-law growth
    axes[1].semilogx(radii, np.log2(doubled/counts), "o-", label=name)
    axes[1].axhline(d, color="gray", linestyle=":", alpha=.5)
axes[0].set(xlabel="Radius r", ylabel="Number of observations within r")
axes[1].set(xlabel="Radius r", ylabel=r"$\log_2[N(2r)/N(r)]$")
for ax in axes:
    ax.set_xticks([.05, .1, .2, .4], labels=["0.05", "0.1", "0.2", "0.4"])
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.legend()
plt.tight_layout(); plt.show()

## 2. Local PCA: approximate a tangent space
Center the data, compute covariance eigenvalues $\lambda_1\ge\cdots\ge\lambda_D$, and select
$$
\widehat d=\min\left\{q:\frac{\sum_{j=1}^q\lambda_j}{\sum_{j=1}^D\lambda_j}\ge .95\right\}.
$$
This is the rule that we saw in the lecture: keep as many components as necessary to explain 95% of the variance. 
In scikit-dimension it requires `lPCA(ver="ratio", alphaRatio=0.95)`; the default is a different rule.
Despite its name, `.fit(X)` fits the supplied cloud globally. Use `.fit_pw(...)` for neighborhoods. Note that the latter returns a vector of local estimates, one per observation, so we need to pool them into a single summary statistic, such as the median.

### Exercise 1 — Local PCA
Predict the global and local answers for the circle, plane and sphere. Complete the local fit and summarize its pointwise estimates with the median. Use the cumulative 95% variance rule.

**API hint:** `skdim.id.lPCA(ver="ratio", alphaRatio=...).fit_pw(X, n_neighbors=...)`
stores the per-point estimates in `.dimension_pw_`. Plotting functions are supplied.

**Question:** Which cloud has global PCA dimension larger than its manifold dimension, and why?

In [ ]:
k = 40
threshold = .95
pca_rows = []
for name, (X, truth) in clouds.items():
    global_fit = skdim.id.lPCA(ver="ratio", alphaRatio=threshold).fit(X)
    # TODO(1): fit local PCA with the threshold and neighborhood size above.
    local_fit = ...
    pca_rows.append({"Cloud": name, "D": X.shape[1], "d": truth,
                     "Global PCA": global_fit.dimension_,
                     "Median local PCA": ...  # TODO(1): median of the pointwise estimates
                    })
print(pd.DataFrame(pca_rows).set_index("Cloud"))

**Your interpretation:**

_Write one or two sentences here._

We now check manually the histograms of the PCA eigenvalues for the sphere and for a local patch around one point.

In [ ]:
_, idx = neighbors(sphere, k)
patch = sphere[np.r_[0, idx[0]]]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True, layout="constrained")

for ax, (name, X) in zip(axes,[("Whole sphere", sphere), ("Local patch", patch)]):
    spectrum = PCA().fit(X).explained_variance_ratio_
    dimension = np.searchsorted(np.cumsum(spectrum), threshold) + 1
    components = np.arange(1, len(spectrum) + 1)

    # Highlight the components needed to reach the variance threshold.
    bars = ax.bar(components, 100 * spectrum, width=0.65, edgecolor="white")

    ax.set(title=(f"{name}," + rf" $\widehat{{d}}={dimension}$ at {threshold:.0%} cumulative variance"),
        xlabel="Principal component",
        xticks=components,
        ylim=(0, 100),
    )
    ax.grid(axis="y", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("Explained variance (%)")
plt.show()

## 3. TwoNN: dimension from distance ratios

Recall that, for each point in our sample, we let $r_1$ and $r_2$ be the distances to the nearest and second-nearest other points in the sample. 
Their ratio $\mu = r_1 / r_2$, under a local homogeneous sampling model, has a cumulative distribution function (CDF) $F(\mu) = 1 - \mu^{-d}$, where $d$ is the intrinsic dimension. 
Hence,
$$d \log t = -\log(1-F_\mu(t)) $$
We will now explore this relationship in practice and compare a manual fit to this law with the `scikit-dimension` implementation of TwoNN.

Note that the empirical CDF 
$$\widehat{F}(\mu) = \frac{1}{n}\sum_{i=1}^n \mathbb{1}_{\{R_i \leq \mu\}} $$
reaches 1 at the largest observation, hence the logarithm $-\log(1-\widehat{F}(\mu))$ is infinite. 

`scikit-dimension` uses plotting positions $i/n$, with $i=1,\ldots,n$, and by default discards the largest 10% of ratios, to exclude the endpoint where the logarithm would be infinite, see the [documentation](https://scikit-dimension.readthedocs.io/en/latest/skdim.id.TwoNN.html#skdim.id.TwoNN).
We use the same convention for an exact comparison.

### Exercise 2 — TwoNN
Complete the distance ratios and the least-squares slope through the origin, the logarithmic coordinates and the library comparison are supplied.

**Hints:** columns 0 and 1 contain $r_1$ and $r_2$; sort the ratios before forming their CDF.
For vectors $x,y$, the slope is $(x^\top y)/(x^\top x)$ (`np.dot`).

**Question:** 
- Is the estimate near the expected sphere dimension? 
- What happens with $\mu$ if we rescale the point clouds? 
- What happens with duplicate observations?
- Repeat this exercise for the circle and plane, and compare `discard=.05`, `.1`, and `.2`.

In [ ]:
X = check_cloud(sphere)
distances, _ = neighbors(X, 2)  # compute matrix of pairwise distances
# TODO(2): compute r2/r1 for every row, then sort the ratios.
mu = ...  # sort the ratios 
discard = .10  # fraction of points to discard
m = int(len(X) * (1 - discard))  # number of points to keep
x_log = np.log(mu[:m])  # x coordinates are the logarithms of the ratios
y_log = -np.log1p(-np.arange(1, m + 1) / len(X))  # compute log(1 + t), with t = -F(mu_i) = -i/n 
# TODO(2): least-squares slope through the origin.
d_manual = ...   # fit a line
twonn = skdim.id.TwoNN(discard_fraction=discard).fit(X)  # use scikit-learn
print(f"Manual slope: {d_manual:.4f}; skdim: {twonn.dimension_:.4f}; true d: 2")

# Plot the empirical CDF and the fitted line
fig, ax = plt.subplots()
ax.scatter(x_log, y_log, s=7, alpha=.5, label="Empirical plotting positions")
ax.plot(x_log, twonn.dimension_ * x_log, color="black", label=f"Fitted slope {twonn.dimension_:.2f}")
ax.set(xlabel="log(r₂/r₁)", ylabel="−log(1−F)", title="TwoNN on a sphere surface")
ax.legend(); plt.show()

**Your interpretation:**

_Write one or two sentences here._

Without discarding the largest ratios, the log-log plot diverges at the right end.

In [ ]:
m = int(len(X))  # now we keep all the points
x_log = np.log(mu[:m])  
y_log = -np.log1p(-np.arange(1, m + 1) / len(X))  
d_manual = np.dot(x_log, y_log) / np.dot(x_log, x_log)   
twonn = skdim.id.TwoNN(discard_fraction=discard).fit(X) 
print(f"Manual slope: {d_manual:.4f}; skdim: {twonn.dimension_:.4f}; true d: 2")

# Plot the empirical CDF and the fitted line
fig, ax = plt.subplots()
ax.scatter(x_log, y_log, s=7, alpha=.5, label="Empirical plotting positions")
ax.plot(x_log, twonn.dimension_ * x_log, color="black", label=f"Fitted slope {twonn.dimension_:.2f}")
ax.set(xlabel="log(r₂/r₁)", ylabel="−log(1−F)", title="TwoNN on a sphere surface")
ax.legend(); plt.show()

## 4. MLE: use the first $k$ neighbors
We now write $R=r_k$ and $u_j=r_j/R$ for $j<k$. We have seen in class that the model gives $F_U(u)=u^d$ and the log-likelihood is then
$$
\ell(d)=(k-1)\log d+(d-1)\sum_{j=1}^{k-1}\log u_j+C,
\qquad
\widehat d_k(x)=\frac{k-1}{\sum_{j=1}^{k-1}\log(r_k/r_j)}.
$$
We set `unbiased=False` to match the lecture's numerator $k-1$, the optional correction uses
$k-2$.

Notice that the MLE is a local estimator, that is, it provides an estimated distance at each point in the point cloud. The `comb` argument controls how to combine these local estimates into a single global estimate. In the subsequent example, we will manually combine these esimates using the arithmetic mean, 
$$\widehat{d}_{k, \mathrm{arithmetic}} = \frac{1}{n}\sum_{i=1}^n \widehat{d}_k(x_i)$$
and compare it to the `comb="mle"` option, which uses the harmonic mean
$$ \widehat{d}_{k, \mathrm{harmonic}} = \left(\frac{1}{n}\sum_{i=1}^n \frac{1}{\widehat{d}_k(x_i)}\right)^{-1}. $$

### Exercise 3 — MLE
Compute one MLE estimate for every observation using the formula above. The distance matrix
has shape `(n, k)` and excludes self-neighbors. Compare your estimates with scikit-dimension,
then run the supplied scale/noise experiment below.

**Hints:** `distances[:, [-1]]` has shape `(n, 1)` and holds $r_k$;
`distances[:, :-1]` holds the first $k-1$ distances. Broadcasting gives all ratios.
Use `np.log(...).sum(axis=1)` for one denominator per observation.

**Question:** Why can a larger neighborhood help with noise but hurt the local geometric approximation?

In [ ]:
def mle_fit(X, k=30):
    return skdim.id.MLE(unbiased=False).fit(check_cloud(X), n_neighbors=k, comb="mle")

k = 30  # number of neighbors to use for the MLE
distances, _ = neighbors(sphere, k)  # compute matrix of pairwise distances
# TODO(3): vectorize the lecture formula; return an array of length len(sphere).
local_manual = ...  # manually compute the local MLE estimates
mle = mle_fit(sphere, k)  # use scikit-dimension

print(f"One point: {local_manual[0]:.3f}")
print(f"Arithmetic mean of local estimates: {local_manual.mean():.3f}")
print(f"Pooled estimate (comb='mle'): {mle.dimension_:.3f}")
# comb='mle' is the harmonic mean, not the arithmetic mean.

# plot results
u = distances[0, :-1] / distances[0, -1]   # normalized distances
d_grid = np.linspace(.3, 5, 250)   # grid of candidate dimensions for the log-likelihood minimization 
log_likelihood = (k-1)*np.log(d_grid) + (d_grid-1)*np.log(u).sum()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
t = np.linspace(0, 1, 100)
for d in [1, 2, 4]: 
    axes[0].plot(t, t**d, label=rf"$d={d}$")
axes[0].step(np.r_[0, u, 1], np.r_[0, np.arange(1, k)/(k-1), 1], where="post", color="black", label="Observed radii")
axes[0].set(xlabel="Normalized radius u", ylabel="CDF"); axes[0].legend()
axes[1].plot(d_grid, log_likelihood - log_likelihood.max())
axes[1].axvline(local_manual[0], color="black", linestyle="--", label="Local MLE")
axes[1].set(xlabel="Candidate dimension d", ylabel="Relative log-likelihood")
axes[1].legend(); plt.tight_layout(); plt.show()

**Your interpretation:**

_Write one or two sentences here._

### Scale and noise: no single neighborhood size is always correct
We reuse the same sphere and add several noise levels, testing what the MLE estimates are for several neighborhood sizes.

In [ ]:
ks = [10, 20, 40, 80, 160]
noise = np.random.default_rng(52).normal(size=sphere.shape)
scale_rows = []
for sigma in [0., .03, .10]:
    X = sphere + sigma * noise
    for k in ks:
        fit = mle_fit(X, k)
        lpca = skdim.id.lPCA(ver="ratio", alphaRatio=.95).fit_pw(X, n_neighbors=k)
        scale_rows.append({"sigma": sigma, "k": k, "MLE": fit.dimension_,
                           "Local PCA (median)": np.median(lpca.dimension_pw_)})
scale_df = pd.DataFrame(scale_rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, method in zip(axes, ["MLE", "Local PCA (median)"]):
    for sigma, group in scale_df.groupby("sigma"):
        ax.plot(group.k, group[method], "o-", label=f"noise σ={sigma}")
    ax.axhline(2, color="gray", linestyle="--")
    ax.set(xlabel="Neighbors k", ylabel="Estimated dimension", title=method)
    ax.legend()
plt.tight_layout(); plt.show()

## 5. Intrinsic dimension of images
### Optional extension: a controlled image generator

**Skip to “Real images” during the session.** This generator is independent of the core.
In the example from the lecture, the authors vary the number of free latent coordinates of a GAN to produce data with different intrinsic dimensions in a controlled manner. 
We now implement a lightweight analogue which renders a Gaussian spot whose horizontal position, vertical position and width can vary.
Hence, while each image has $16 \times 16 = 256$ pixels, it has at most three free generating parameters.

In [ ]:
def spot_images(latents, side=16):
    grid = np.linspace(-1, 1, side)
    gx, gy = np.meshgrid(grid, grid)
    cx = .5 * latents[:, 0, None, None] # horizontal position
    cy = .5 * latents[:, 1, None, None] # vertical position
    width = .22 + .05 * latents[:, 2, None, None] # width
    return np.exp(-((gx-cx)**2 + (gy-cy)**2)/(2*width**2))

latent = np.random.default_rng(53).uniform(-1, 1, (900, 3)) # sample 900 latent coordinates in [-1,1]^3
image_rows = []
fig, axes = plt.subplots(3, 5, figsize=(8, 5))
for d in [1, 2, 3]:
    z = latent.copy(); z[:, d:] = 0 # we only vary the first d coordinates, the rest are fixed to 0
    images = spot_images(z) # create the images with those free parameters
    X = images.reshape(len(images), -1)
    image_rows.append({"Free parameters": d, "D": X.shape[1],
                       "TwoNN": skdim.id.TwoNN().fit(X).dimension_,
                       "MLE (k=20)": mle_fit(X, 20).dimension_})
    for j, ax in enumerate(axes[d-1]):
        ax.imshow(images[j], cmap="gray", vmin=0, vmax=1); ax.axis("off")
        if j == 0: ax.set_title(f"{d} free parameter(s)")
plt.tight_layout(); plt.show()
display(pd.DataFrame(image_rows))

### Real images: handwritten digits
We now emulate the example from the lecture, which measures the intrinsic dimension of common datasets. 
We use `load_digits`, a dataset of handwritten digits bundled with scikit-learn, with $8 \times 8$ grayscale images. Note that this is not the MNIST dataset.
We normalize all pixel values by it maximum value (16), to have one common scale. 
Note that the true generating dimension of this example is unknown.

### Exercise 4 — Real images
Use scikit-dimension's MLE on the digit images for each supplied neighborhood size. Store its
**pooled** dimension in the results table. Data loading and the TwoNN comparison are supplied.

**API hint:** `skdim.id.MLE(unbiased=False).fit(X, n_neighbors=k, comb="mle")`;
read the scalar `.dimension_` from the fitted object.

**Question:** What is D? How does the estimate change with k? Can we call any value the true d?

In [ ]:
# load dataset and normalize pixel values to [0,1]
digits = load_digits()
X_digits = digits.data.astype(float) / 16.0
y_digits = digits.target
print("Observations, pixel coordinates:", X_digits.shape)
check_cloud(X_digits)

# plot some examples
fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_digits[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.set_title(str(y_digits[i])); ax.axis("off")
plt.tight_layout(); plt.show()

# compute MLE for several values of k, and TwoNN
digit_rows = []
for k in [10, 20, 40, 80]:
    # TODO(4): fit MLE to X_digits using the current k and harmonic aggregation.
    fit = ...
    # TODO(4): extract the pooled estimate, not the array of local estimates.
    digit_rows.append({"k": k, "MLE": ...})
display(pd.DataFrame(digit_rows))
print("TwoNN:", round(skdim.id.TwoNN().fit(X_digits).dimension_, 2))

**Your interpretation:**

_Write one or two sentences here._

### Optional extension: change the ambient dimension without adding information

**Skip to section 7 during the session.** The remaining blocks in sections 5 and 6 are optional.
Nearest-neighbor upsampling repeats pixels. Repeating each pixel $s^2$ times multiplies all
Euclidean distances by $s$, so all neighbor-distance ratios remain unchanged. This is a particularly
controlled way to raise $D$ without adding degrees of freedom.

In [ ]:
X_small = X_digits[:500]
upsampled = np.repeat(np.repeat(X_small.reshape(-1, 8, 8), 2, axis=1), 2, axis=2).reshape(-1, 256)
d0, _ = neighbors(X_small, 20)
d1, _ = neighbors(upsampled, 20)
np.testing.assert_allclose(d1, 2*d0, atol=1e-10)
display(pd.DataFrame([
    {"Representation": name, "D": X.shape[1], "MLE": mle_fit(X, 20).dimension_}
    for name, X in [("8 × 8 pixels", X_small), ("16 × 16 repeated pixels", upsampled)]
]))

## 6. Optional: intrinsic dimension of internal representations

**After class.** This section is self-contained after the setup and digit-data cells. Skip to section 7 for the final core exercise.
Train a small MLP on digits and measure the intrinsic dimensions of the same held-out observations at each layer, after one epoch
and after 80 epochs. In this example, the widths of the MLPs are ambient dimensions; they are not intrinsic dimensions of data.
The output cloud consists of **logits** (before softmax).
The lecture's CNN/transformer “hunchback” is an empirical finite-scale pattern. This small MLP does not exhibit it.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=.3, stratify=y_digits, random_state=SEED)
net = MLPClassifier(hidden_layer_sizes=(64, 32, 16), activation="tanh",
                    solver="adam", batch_size=128, learning_rate_init=.003,
                    alpha=1e-4, random_state=SEED)
# partial_fit initializes and performs one epoch; track this as EARLY, not untrained.
net.partial_fit(X_train, y_train, classes=np.arange(10))

def representations(model, X):
    result = {"Input": X}
    H = X
    for j, (W, b) in enumerate(zip(model.coefs_[:-1], model.intercepts_[:-1]), 1):
        H = np.tanh(H @ W + b)
        result[f"Hidden {j}"] = H
    result["Logits"] = H @ model.coefs_[-1] + model.intercepts_[-1]
    return result

early = representations(net, X_test)
accuracy = [net.score(X_test, y_test)]
for epoch in range(79):
    net.partial_fit(X_train, y_train)
    accuracy.append(net.score(X_test, y_test))
trained = representations(net, X_test)
print(f"After 80 epochs: train accuracy={net.score(X_train, y_train):.3f}, test accuracy={accuracy[-1]:.3f}")
# Verify the manually extracted network output against its public prediction API.
predictions = net.classes_[trained["Logits"].argmax(axis=1)]
np.testing.assert_array_equal(predictions, net.predict(X_test))

We compare the pooled held-out cloud with point clouds of equal size in each of the classes, closer to the per-category analysis shown in the lecture. 
The across-class standard deviation measures the heterogeneity of the intrinsic dimension estimates across classes, not uncertainty. 
Small class samples make the estimates noisy.

In [ ]:
per_class_n = min(np.bincount(y_test))
class_indices = [np.flatnonzero(y_test == c)[:per_class_n] for c in range(10)]
representation_rows = []
for stage, reps in [("After 1 epoch", early), ("After 80 epochs", trained)]:
    for layer, H in reps.items():
        check_cloud(H)
        by_class = [skdim.id.TwoNN().fit(check_cloud(H[idx])).dimension_ for idx in class_indices]
        representation_rows.append({"Stage": stage, "Layer": layer, "D": H.shape[1],
                                    "Pooled TwoNN": skdim.id.TwoNN().fit(H).dimension_,
                                    "Class mean": np.mean(by_class), "Class SD": np.std(by_class)})
representation_df = pd.DataFrame(representation_rows)
display(representation_df.round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for stage, group in representation_df.groupby("Stage", sort=False):
    axes[0].plot(group.Layer, group["Pooled TwoNN"], "o-", label=stage)
    axes[1].errorbar(group.Layer, group["Class mean"], yerr=group["Class SD"], marker="o", capsize=3, label=stage)
for ax, title in zip(axes, ["Pooled held-out observations", f"Class-conditional: {per_class_n} images/class"]):
    ax.set(title=title, ylabel="TwoNN estimate")
    ax.tick_params(axis="x", rotation=25); ax.legend()
plt.tight_layout(); plt.show()

**TO DO.** Compare estimates after 1, 20 and 80 epochs, or change the hidden widths.
Does accuracy necessarily improve when estimated dimension decreases? One model and one seed cannot
establish such a relationship. For transformer activations shaped `(examples, tokens, features)`,
mean-pooling over tokens gives one vector per example, as in the lecture; flattening tokens instead
would define a different point cloud.

## 7. The union of manifolds hypothesis
A single pooled estimate can conceal components with different dimensions, we now test this in practice. C
We construct a separated circle ($d=1$) and sphere surface ($d=2$) in the same ambient space $D=3$.
In this example, both the globality of an estimator and sufficiently local sampling matter. We will see that global estimators give us a sole estimated dimension between 1 and 2, while the local MLE can recover the two different dimensions in the two components.

### Exercise 5 — A union of manifolds
Run the supplied mixture visualization, then complete the final loop in the same cell to estimate
the circle and sphere dimensions **separately**. Reuse `mle_fit(X, k)`, which returns a fitted
scikit-dimension estimator, and select its pooled scalar.

**Question:** Compare the two component estimates with the pooled result. Does the latter describe
either component?

In [ ]:
rg = np.random.default_rng(55)
a = rg.uniform(0, 2*np.pi, 600)
component1 = np.c_[np.cos(a)-3, np.sin(a), np.zeros(len(a))] # circle in the plane x1-x2, shifted to the left
component2 = rg.normal(size=(600, 3)) 
component2 /= np.linalg.norm(component2, axis=1, keepdims=True) # sphere surface, centered at the origin
component2 += [3, 0, 0] # shift the sphere to the right
mixture = np.vstack([component1, component2]) # combine the two components into a single dataset
component_labels = np.repeat([1, 2], 600)
mixed_fit = mle_fit(mixture, 30)

# plot pointclouds and the MLE estimates for each component
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(121, projection="3d")
p = ax.scatter(*mixture.T, c=mixed_fit.dimension_pw_, s=6, cmap="viridis", vmin=.5, vmax=3)
ax.set(title="Local MLE on a union", xlabel="r$x_1$", ylabel=r"$x_2$", zlabel=r"$x_3$")
fig.colorbar(p, ax=ax, label="Local dimension", shrink=.65, pad=.18)
ax = fig.add_subplot(122)
for d in [1, 2]:
    ax.hist(mixed_fit.dimension_pw_[component_labels == d], bins=30, alpha=.6, label=f"Component d={d}")
ax.set(xlabel="Local MLE estimate", ylabel="Observations"); ax.legend()
plt.tight_layout(); plt.show()
print(f"Pooled MLE: {mixed_fit.dimension_:.2f}; pooled TwoNN: {skdim.id.TwoNN().fit(mixture).dimension_:.2f}")

component_rows = []
for name, X in [("Circle", component1), ("Sphere surface", component2)]:
    # TODO(5): fit MLE to this component and extract the pooled dimension.
    component_id = ...
    component_rows.append({"Component": name, "MLE": component_id})
display(pd.DataFrame(component_rows))

**Your interpretation:**

_Write one or two sentences here._

### Optional extension: class-conditional dimensions of digit images

**Skip to the recap if time is up.**
Match class sample counts so that class comparisons are less confounded by density. Use several
values of `k`. Equal sample sizes do not remove differences in shape, density, noise or boundary effects.

In [ ]:
per_class = min(np.bincount(y_digits))
rg = np.random.default_rng(56)
digit_groups = {c: X_digits[rg.choice(np.flatnonzero(y_digits == c), per_class, replace=False)] for c in range(10)}
class_rows = []
for c, X in digit_groups.items():
    for k in [10, 20, 40]:
        class_rows.append({"Digit": c, "k": k, "MLE": mle_fit(X, k).dimension_})
class_df = pd.DataFrame(class_rows)
fig, ax = plt.subplots()
for k, group in class_df.groupby("k"):
    ax.plot(group.Digit, group.MLE, "o-", label=f"k={k}")
ax.set(xlabel="Digit class", ylabel="Pooled MLE within class", xticks=range(10),
       title=f"Equal sample count: {per_class} images per class")
ax.legend(); plt.show()

**TO DO.** Move the synthetic components closer and increase `k` until neighborhoods mix.
Does the pooled estimate describe either component? Do class-dependent digit estimates prove that
the population support is disconnected? No: these are finite-sample geometric diagnostics;
disconnectedness is a separate claim, and class labels need not coincide with connected components.
The lecture investigates that claim with generative models; we do not train WAE/D-WAE models here.